# BERT Multi-Class Text Classification Pipeline

End-to-end pipeline for fine-tuning **`bert-base-cased`** on multi-class text classification.

**Architecture:**

| Class | Responsibility |
|---|---|
| `Config` | Centralised hyperparameters (dataclass) |
| `DataProcessor` | Loading, filtering, encoding, tokenization, DataLoader creation |
| `BERTTrainer` | Model initialisation, optimizer/scheduler setup, training, evaluation |
| `Pipeline` | Orchestrates the full workflow via a single `.run()` call |



## 1 · Setup

In [1]:
# Uncomment on first run / Colab
# !pip install -q transformers datasets scikit-learn torch pandas openpyxl evaluate accelerate tqdm

In [2]:
import os
import pickle
import shutil
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    f1_score,
    confusion_matrix,   )

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    get_scheduler,
)
from torch.utils.data import DataLoader
from torch.optim import AdamW, Adagrad, Adadelta, RMSprop
from torch.optim.lr_scheduler import (
    CosineAnnealingLR,
    PolynomialLR,
    ExponentialLR,
)
import matplotlib.pyplot as plt
import seaborn as sns

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

Device: cuda


In [3]:
from google.colab import files
uploaded = files.upload()  # drag & drop bbc_news_full.csv

Saving bbc_news_full.csv to bbc_news_full (1).csv


## 2 · Configuration

All hyperparameters live in a single `@dataclass`. Change values here — the rest of the notebook picks them up automatically.

In [4]:
@dataclass
class Config:
    """Centralised configuration for the entire pipeline."""

    # ── Data ──
    data_path:   str = "bbc_news_full.csv"
    text_col:    str = "text"
    label_col:   str = "label_text"
    test_size: float = 0.2

    # ── Model ──
    checkpoint:  str = "bert-base-cased"

    # ── Training ──
    epochs:      int = 3
    batch_size:  int = 16
    learning_rate: float = 2e-5
    optimizer:   str = "AdamW"             # AdamW | RMSprop | Adagrad | Adadelta
    lr_scheduler: str = "linear"           # linear | cosine | polynomial | exponential

    # ── Output ──
    save_dir:    str = "bert_multiclass_model"


cfg = Config()
print(cfg)

Config(data_path='bbc_news_full.csv', text_col='text', label_col='label_text', test_size=0.2, checkpoint='bert-base-cased', epochs=3, batch_size=16, learning_rate=2e-05, optimizer='A[...]


## 3 · DataProcessor

Handles everything between raw CSV and ready-to-train DataLoaders:
- File loading (CSV / Excel)
- Class filtering (optional subset of labels)
- Label encoding (`LabelEncoder` → integer labels)
- Tokenization & DataLoader construction

In [5]:
class DataProcessor:
    """Load, encode, tokenize, and batch text data for BERT."""

    def __init__(self, cfg: Config):
        self.cfg = cfg
        self.label_encoder = LabelEncoder()
        self.tokenizer = AutoTokenizer.from_pretrained(cfg.checkpoint)
        self.collator = DataCollatorWithPadding(tokenizer=self.tokenizer)

    # ── Public API ──

    def load(self, allowed_classes=None):
        """
        Load dataset, filter classes, encode labels, and train/test split.

        Returns:
            x_train, x_test  – text arrays (shape N,1)
            y_train, y_test  – integer label arrays (shape N,)
        """
        df = self._read_file(self.cfg.data_path)
        df = df.dropna(subset=[self.cfg.text_col, self.cfg.label_col]).reset_index(drop=True)
        df = self._filter_classes(df, allowed_classes)

        x = df[self.cfg.text_col].values.reshape(-1, 1)
        y = self.label_encoder.fit_transform(df[self.cfg.label_col])

        x_train, x_test, y_train, y_test = train_test_split(
            x, y,
            test_size=self.cfg.test_size,
            stratify=y,
            random_state=0,
        )

        print(f"  Classes: {self.num_classes}  |  Train: {len(x_train):,}  |  Test: {len(x_test):,}")
        print(f"  Labels: {list(self.label_encoder.classes_)}")
        return x_train, x_test, y_train, y_test

    def build_dataloaders(self, x_train, x_test, y_train, y_test):
        """
        Tokenize splits and wrap them in DataLoaders.

        Returns:
            train_dl, test_dl
        """
        train_ds = self._tokenize(x_train, y_train)
        test_ds  = self._tokenize(x_test, y_test)

        train_dl = DataLoader(train_ds, batch_size=self.cfg.batch_size, collate_fn=self.collator, shuffle=True)
        test_dl  = DataLoader(test_ds,  batch_size=self.cfg.batch_size, collate_fn=self.collator)

        print(f"  Train batches: {len(train_dl)}  |  Test batches: {len(test_dl)}")
        return train_dl, test_dl

    @property
    def num_classes(self):
        """Number of classes discovered during load()."""
        return len(self.label_encoder.classes_)

    @property
    def class_names(self):
        """Array of class name strings."""
        return self.label_encoder.classes_

    # ── Private helpers ──

    @staticmethod
    def _read_file(path: str) -> pd.DataFrame:
        if path.endswith(".xlsx") or path.endswith(".xls"):
            return pd.read_excel(path)
        return pd.read_csv(path)

    def _filter_classes(self, df, allowed_classes):
        if allowed_classes is None:
            return df
        mask = df[self.cfg.label_col].isin(allowed_classes)
        print(f"  Filtered {mask.sum():,}/{len(df):,} rows → {len(allowed_classes)} active classes")
        return df[mask].reset_index(drop=True)

    def _tokenize(self, texts, labels):
        flat = [t for row in texts for t in row]
        encoded = {"input_ids": [], "token_type_ids": [], "attention_mask": [], "label": labels.tolist()}
        for text in flat:
            tok = self.tokenizer(text, truncation=True, padding=True)
            encoded["input_ids"].append(tok["input_ids"])
            encoded["token_type_ids"].append(tok["token_type_ids"])
            encoded["attention_mask"].append(tok["attention_mask"])
        return Dataset.from_dict(encoded)

## 4 · BERTTrainer

Encapsulates the model lifecycle:
- Model loading with configurable optimizer & LR scheduler
- Training loop with progress tracking
- Evaluation with classification report
- Inference on raw text
- Model saving & archiving

In [6]:
class BERTTrainer:
    """Train, evaluate, and run inference with a BERT classification model."""

    OPTIMIZERS = {
        "AdamW":    AdamW,
        "RMSprop":  RMSprop,
        "Adagrad":  Adagrad,
        "Adadelta": Adadelta,
    }

    def __init__(self, cfg: Config, num_classes: int):
        self.cfg = cfg
        self.model = AutoModelForSequenceClassification.from_pretrained(
            cfg.checkpoint, num_labels=num_classes,
        ).to(DEVICE)
        self.history = []

    # ── Setup ──

    def setup(self, train_dl):
        """
        Configure optimizer and LR scheduler based on Config.

        Must be called before train().
        """
        self.total_steps = self.cfg.epochs * len(train_dl)
        self.optimizer = self._build_optimizer()
        self.scheduler = self._build_scheduler()

        print(f"  Optimizer: {self.cfg.optimizer}  |  Scheduler: {self.cfg.lr_scheduler}")
        print(f"  Total training steps: {self.total_steps:,}")

    # ── Training ──

    def train(self, train_dl, eval_dl=None, class_names=None):
        progress = tqdm(range(self.total_steps), desc="Training")

        for epoch in range(1, self.cfg.epochs + 1):
            self.model.train()
            epoch_loss = 0.0
            for batch in train_dl:
                batch = {k: v.to(DEVICE) for k, v in batch.items()}
                outputs = self.model(**batch)

                outputs.loss.backward()
                self.optimizer.step()
                self.scheduler.step()
                self.optimizer.zero_grad()

                epoch_loss += outputs.loss.item()
                progress.update(1)

            train_loss = epoch_loss / len(train_dl)
            record = {"epoch": epoch, "train_loss": train_loss}

            # Validation after each epoch
            if eval_dl is not None:
                val_metrics = self._compute_metrics(eval_dl)
                record.update(val_metrics)
                print(f"  Epoch {epoch}/{self.cfg.epochs}  |  "
                      f"train_loss = {train_loss:.4f}  |  "
                      f"val_loss = {val_metrics['val_loss']:.4f}  |  "
                      f"val_acc = {val_metrics['val_acc']:.4f}  |  "
                      f"val_f1 = {val_metrics['val_f1']:.4f}")
            else:
                print(f"  Epoch {epoch}/{self.cfg.epochs}  |  train_loss = {train_loss:.4f}")

            self.history.append(record)

    # ── Evaluation ──

    def evaluate(self, eval_dl, class_names):
        """
        Run inference on eval set and return classification report string.
        """
        self.model.eval()
        all_preds, all_trues = [], []

        for batch in eval_dl:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            with torch.no_grad():
                outputs = self.model(**batch)

            all_preds.extend(torch.argmax(outputs.logits, dim=-1).cpu().numpy())
            all_trues.extend(batch["labels"].cpu().numpy())
            self.y_true = all_trues
            self.y_pred = all_preds

        return classification_report(
            all_trues, all_preds,
            target_names=class_names,
            zero_division=0,
        )

    # ── Inference ──

    def predict(self, texts, data_processor: 'DataProcessor'):
        """
        Predict class labels for a list of raw text strings.

        Returns:
            numpy array of predicted class name strings
        """
        encoded = {"input_ids": [], "token_type_ids": [], "attention_mask": []}
        for text in texts:
            tok = data_processor.tokenizer(text, truncation=True, padding=True)
            encoded["input_ids"].append(tok["input_ids"])
            encoded["token_type_ids"].append(tok["token_type_ids"])
            encoded["attention_mask"].append(tok["attention_mask"])

        ds = Dataset.from_dict(encoded)
        dl = DataLoader(ds, batch_size=self.cfg.batch_size, collate_fn=data_processor.collator)

        self.model.eval()
        all_preds = []
        for batch in dl:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            with torch.no_grad():
                logits = self.model(**batch).logits
            all_preds.extend(torch.argmax(logits, dim=-1).cpu().numpy())

        return data_processor.class_names[all_preds]

    # ── Save ──

    def save(self, report: str, tokenizer):
        """
        Save model weights, tokenizer, classification report, and create a zip archive.
        """
        save_dir = self.cfg.save_dir
        os.makedirs(save_dir, exist_ok=True)

        self.model.save_pretrained(save_dir)
        tokenizer.save_pretrained(save_dir)

        with open(os.path.join(save_dir, "classification_report.pkl"), "wb") as f:
            pickle.dump(report, f)

        shutil.make_archive(save_dir, "zip", save_dir)

        print(f"  Model saved → {save_dir}/")
        print(f"  Archive     → {save_dir}.zip")

    # ── Private helpers ──

    def _build_optimizer(self):
        name = self.cfg.optimizer
        if name not in self.OPTIMIZERS:
            raise ValueError(f"Unsupported optimizer: {name}. Choose from {list(self.OPTIMIZERS)}")
        return self.OPTIMIZERS[name](self.model.parameters(), lr=self.cfg.learning_rate)

    def _build_scheduler(self):
        schedulers = {
            "linear":      lambda: get_scheduler("linear", optimizer=self.optimizer,
                                                 num_warmup_steps=4,
                                                 num_training_steps=self.total_steps),
            "cosine":      lambda: CosineAnnealingLR(self.optimizer, T_max=self.total_steps),
            "polynomial":  lambda: PolynomialLR(self.optimizer, total_iters=self.total_steps, power=2),
            "exponential": lambda: ExponentialLR(self.optimizer, gamma=0.1),
        }
        name = self.cfg.lr_scheduler
        if name not in schedulers:
            raise ValueError(f"Unsupported scheduler: {name}. Choose from {list(schedulers)}")
        return schedulers[name]()
    @torch.no_grad()
    def _compute_metrics(self, eval_dl):
        self.model.eval()
        total_loss = 0.0
        all_preds, all_trues = [], []

        for batch in eval_dl:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            outputs = self.model(**batch)
            total_loss += outputs.loss.item() * batch["labels"].size(0)
            all_preds.extend(torch.argmax(outputs.logits, dim=-1).cpu().numpy())
            all_trues.extend(batch["labels"].cpu().numpy())

        return {
            "val_loss": total_loss / len(eval_dl.dataset),
            "val_acc":  accuracy_score(all_trues, all_preds),
            "val_f1":   f1_score(all_trues, all_preds, average="macro", zero_division=0),
        }

## 5 · Pipeline

Orchestrator class — wires `DataProcessor` and `BERTTrainer` together into a single `.run()` call.

In [7]:
class Pipeline:
    """End-to-end orchestrator: data → train → evaluate → save."""

    def __init__(self, cfg: Config):
        self.cfg = cfg
        self.data_processor = DataProcessor(cfg)
        self.trainer = None      # created after data is loaded (needs num_classes)
        self.report = None

    def run(self, allowed_classes=None):
        """
        Execute the full pipeline.

        Args:
            allowed_classes: optional list of class names to train on (default: all)

        Returns:
            self (for chaining, e.g. pipeline.run().predict(...))
        """
        # Step 1 — Data
        print("Step 1/6 · Loading data...")
        x_train, x_test, y_train, y_test = self.data_processor.load(allowed_classes)

        # Step 2 — DataLoaders
        print("\nStep 2/6 · Preparing dataloaders...")
        train_dl, test_dl = self.data_processor.build_dataloaders(
            x_train, x_test, y_train, y_test,
        )
        self.test_dl = test_dl

        # Step 3 — Model setup
        print("\nStep 3/6 · Setting up model...")
        self.trainer = BERTTrainer(self.cfg, self.data_processor.num_classes)
        self.trainer.setup(train_dl)

        # Step 4 — Training
        print("\nStep 4/6 · Training...")
        self.trainer.train(train_dl, eval_dl=test_dl, class_names=self.data_processor.class_names)

        # Step 5 — Evaluation
        print("\nStep 5/6 · Evaluating...")
        self.report = self.trainer.evaluate(test_dl, self.data_processor.class_names)
        print(self.report)

        # Step 6 — Save
        print("Step 6/6 · Saving model...")
        self.trainer.save(self.report, self.data_processor.tokenizer)

        print("\n" + "=" * 50)
        print("  Pipeline complete.")
        print("=" * 50)
        return self

    def predict(self, texts):
        """
        Classify new texts using the trained model.

        Args:
            texts: list[str]

        Returns:
            numpy array of predicted class name strings
        """
        if self.trainer is None:
            raise RuntimeError("Call .run() before .predict()")
        return self.trainer.predict(texts, self.data_processor)

## 6 · Visualiser

Dedicated class for plotting training results. Reads from a trained `Pipeline`

- `plot_loss()` — standalone loss curve
- `plot_metrics()` — standalone accuracy & F1 curve
- `plot_confusion_matrix()` — standalone confusion matrix

In [8]:
class Visualiser:
    """Plot learning curves and confusion matrix from a trained Pipeline."""

    def __init__(self, pipeline: Pipeline):
        self.pipeline = pipeline
        self.trainer = pipeline.trainer
        self.class_names = pipeline.data_processor.class_names

    def plot_all(self, save_path="training_results.png"):
        """Three-panel figure: loss, metrics, confusion matrix."""
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))

        self._plot_loss(axes[0])
        self._plot_metrics(axes[1])
        self._plot_confusion_matrix(axes[2])

        plt.tight_layout()
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        plt.show()
        print(f"  Saved → {save_path}")

    def plot_loss(self):
        """Standalone loss curve."""
        fig, ax = plt.subplots(figsize=(6, 4))
        self._plot_loss(ax)
        plt.tight_layout()
        plt.show()

    def plot_metrics(self):
        """Standalone accuracy & F1 curve."""
        fig, ax = plt.subplots(figsize=(6, 4))
        self._plot_metrics(ax)
        plt.tight_layout()
        plt.show()

    def plot_confusion_matrix(self):
        """Standalone confusion matrix."""
        fig, ax = plt.subplots(figsize=(6, 5))
        self._plot_confusion_matrix(ax)
        plt.tight_layout()
        plt.show()

    # ── Private helpers ──

    def _get_history(self):
        return pd.DataFrame(self.trainer.history)

    def _plot_loss(self, ax):
        h = self._get_history()
        ax.plot(h["epoch"], h["train_loss"], "o-", label="Train Loss", linewidth=2)
        if "val_loss" in h:
            ax.plot(h["epoch"], h["val_loss"], "s-", label="Val Loss", linewidth=2)
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Loss")
        ax.set_title("Loss Curve")
        ax.legend()
        ax.grid(True, alpha=0.3)

    def _plot_metrics(self, ax):
        h = self._get_history()
        if "val_acc" in h:
            ax.plot(h["epoch"], h["val_acc"], "o-", label="Val Accuracy", linewidth=2)
            ax.plot(h["epoch"], h["val_f1"],  "s-", label="Val F1 (macro)", linewidth=2)
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Score")
        ax.set_title("Validation Metrics")
        ax.set_ylim(0, 1.05)
        ax.legend()
        ax.grid(True, alpha=0.3)

    def _plot_confusion_matrix(self, ax):
        cm = confusion_matrix(self.trainer.y_true, self.trainer.y_pred)
        sns.heatmap(
            cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=self.class_names,
            yticklabels=self.class_names,
            ax=ax,
        )
        ax.set_xlabel("Predicted")
        ax.set_ylabel("True")
        ax.set_title("Confusion Matrix")

---
## 7 · Run Pipeline

In [9]:
pipeline = Pipeline(cfg)
pipeline.run()

# To visualise results:
# vis = Visualiser(pipeline)
# vis.plot_all()


## 8 · Inference Example

In [ ]:
# Test predictions on new texts
test_texts = [
    "Apple's latest iPhone release is revolutionary",
    "The match was thrilling with amazing goals",
    "New government policy announced today"
]

predictions = pipeline.predict(test_texts)
for text, pred in zip(test_texts, predictions):
    print(f"Text: {text}")
    print(f"Prediction: {pred}\n")